# 2. Descriptive Statistics & Exploratory Analysis

## Overview
This notebook generates comprehensive descriptive statistics about the German EV charging infrastructure, analyzing:

1. **Connector distribution** by federal state and year (normal vs. fast chargers)
2. **Power capacity** analysis — total and average power per state/year
3. **Connector type trends** — growth of different plug standards over time
4. **Operator analysis** — top operators by charging stations and connection points
5. **Cumulative growth visualizations** — chargers, power ranges, connector types

All results are exported to the `results/figures/statistics/` directory.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Ensure output directory exists
os.makedirs('../results/figures/statistics', exist_ok=True)

## 2.1 Load and Prepare Data

In [ ]:
# Load the cleaned charging station dataset
data = pd.read_csv('../data/processed/ChargingStationCleaned.csv')

# Extract the commissioning year
data['year'] = pd.to_datetime(data['commissioning_date']).dt.year

# Fix encoding artifacts in federal state names (German umlaut characters)
data['federal_state'] = data['federal_state'].replace({
    'Baden-Wï¿½rttemberg': 'Baden-Württemberg',
    'Thï¿½ringen': 'Thüringen'
})

print(f'Dataset: {len(data)} charging stations, {data["year"].nunique()} unique years')
print(f'Year range: {data["year"].min()} - {data["year"].max()}')
data.columns.tolist()

## 2.2 Analysis 1: Connector Count by Federal State & Year

Compute cumulative connector counts per federal state, separated by charger type (normal vs. fast), with year-over-year percentage growth.

In [ ]:
# Count connectors by federal state, year, and charger type
conn_by_land_year = data.groupby(['federal_state', 'year', 'type_of_charger']).size().reset_index(name='count')

# Compute cumulative totals per state and charger type
conn_by_land_year['cumulative_count'] = (
    conn_by_land_year.groupby(['federal_state', 'type_of_charger'])['count'].cumsum()
)

# Year-over-year percentage change
conn_by_land_year['pct_change'] = (
    conn_by_land_year.groupby(['federal_state', 'type_of_charger'])['cumulative_count'].pct_change() * 100
)

# Export to Excel
conn_by_land_year.to_excel('../results/figures/statistics/point_one.xlsx', index=False)
print('Connector counts by state and year (cumulative):')
conn_by_land_year.head(10)

## 2.3 Analysis 2: Power Capacity by Federal State & Year

Track the total installed power capacity (kW) and average power per station across states and years.

In [ ]:
# Aggregate total and mean power by state, year, and charger type
power_by_land_year = data.groupby(['federal_state', 'year', 'type_of_charger']).agg(
    {'power_connection_[kw]': ['sum', 'mean']}
).reset_index()
power_by_land_year.columns = ['federal_state', 'year', 'type_of_charger', 'total_power', 'avg_power']

# Cumulative total power
power_by_land_year['cumulative_total_power'] = (
    power_by_land_year.groupby(['federal_state', 'type_of_charger'])['total_power'].cumsum()
)

# Year-over-year growth rate
power_by_land_year['pct_change_total_power'] = (
    power_by_land_year.groupby(['federal_state', 'type_of_charger'])['cumulative_total_power'].pct_change() * 100
)

power_by_land_year.to_excel('../results/figures/statistics/point_two.xlsx', index=False)
print('Power capacity by state and year (cumulative):')
power_by_land_year.head(10)

## 2.4 Analysis 3: Connector Type Trends Over Time

Track the adoption of different plug standards (Type 2 AC, CCS Combo, CHAdeMO, etc.) across years.

In [ ]:
# Unify all connector columns into a single long-format DataFrame
connector_columns = ['type_of_plug_1', 'type_of_plug_2', 'type_of_plug_3', 'type_of_plug_4']
data_melted = pd.melt(
    data, id_vars=['year'], value_vars=connector_columns,
    var_name='connector_column', value_name='connector_type'
).dropna()

# Handle multi-plug entries (comma-separated plug types in a single field)
data_melted['connector_type'] = data_melted['connector_type'].apply(lambda x: x.split(', '))
data_melted = data_melted.explode('connector_type')

# Compute cumulative count for each connector type per year
connector_by_year = (
    data_melted.groupby(['year', 'connector_type']).size()
    .groupby('connector_type').cumsum()
    .reset_index(name='count')
)

# Year-over-year growth rate per connector type
connector_by_year['pct_change'] = (
    connector_by_year.groupby('connector_type')['count'].pct_change() * 100
)

connector_by_year.to_excel('../results/figures/statistics/point_three.xlsx', index=False)
print('Connector type distribution by year (cumulative):')
connector_by_year.head(10)

## 2.5 Visualization: Cumulative Charging Station Growth

Bar chart showing the cumulative number of normal and fast charging stations from 2009-2022.

In [ ]:
# Pivot: year × charger_type → count
hist_data = data.pivot_table(
    index='year', columns='type_of_charger',
    values='power_connection_[kw]', aggfunc='count'
).fillna(0)

# Compute cumulative sums and reindex to include all years from 2009
hist_data_cumulative = hist_data.cumsum()
hist_data_cumulative = hist_data_cumulative.reindex(
    range(2009, hist_data_cumulative.index.max() + 1)
).fillna(method='ffill')

# Plot
plt.rcParams['font.size'] = 14
fig, ax = plt.subplots(figsize=(20, 6))
hist_data_cumulative.plot(kind='bar', stacked=False, ax=ax)
plt.xlabel('Year')
plt.ylabel('Number of Stations')
plt.legend(title='Type of Charger')

# Annotate bars with counts
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() * 1.005, p.get_height() + 150))

plt.savefig('../results/figures/statistics/cumulative_normal_vs_fast.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.6 Visualization: Total Cumulative Charging Stations in Germany

In [ ]:
# Cumulative total charging stations across all of Germany
total_by_year = data['year'].value_counts().sort_index().cumsum()
total_by_year = total_by_year.reindex(
    range(2009, total_by_year.index.max() + 1)
).fillna(method='ffill')

plt.figure(figsize=(12, 6))
ax = total_by_year.plot(kind='bar')
plt.xlabel('Year')
plt.ylabel('Number of Stations')

for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() * 1.005, p.get_height() + 150))

plt.savefig('../results/figures/statistics/total_cumulative_stations.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.7 Visualization: Power Range Distribution Over Time

Stacked bar chart showing the distribution of charging stations across power output categories.

In [ ]:
# Define power output bins (in kW)
bins = [0, 3.7, 15, 22, 49, 59, 149, 299, np.inf]
labels = ['0-3.7', '3.7-15', '15-22', '22-49', '49-59', '59-149', '149-299', '>299']

data['power_range'] = pd.cut(data['power_connection_[kw]'], bins=bins, labels=labels)

# Cumulative count per power range and year
power_range_by_year = data.groupby(['year', 'power_range']).size().unstack().fillna(0).cumsum()
power_range_by_year = power_range_by_year.reindex(
    range(2009, power_range_by_year.index.max() + 1)
).fillna(method='ffill')

plt.rcParams['font.size'] = 14
ax = power_range_by_year.plot(kind='bar', stacked=True)
plt.xlabel('Year')
plt.ylabel('Number of Stations')
plt.legend(title='Power Range (kW)')
plt.savefig('../results/figures/statistics/power_range_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 2.8 Visualization: Connector Type Clusters Over Time

Group the many connector subtypes into major categories for clearer visualization.

In [ ]:
def cluster_connector(connector):
    """Map specific connector types to broader categories."""
    if 'AC Steckdose Typ 2' in connector:
        return 'AC Steckdose Typ 2'
    elif 'AC Kupplung Typ 2' in connector:
        return 'AC Kupplung Typ 2'
    elif 'DC Kupplung Combo' in connector:
        return 'DC Kupplung Combo'
    elif 'AC Schuko' in connector:
        return 'AC Schuko'
    elif 'DC CHAdeMO' in connector:
        return 'DC CHAdeMO'
    else:
        return 'Other'

def split_connector(connector):
    """Split comma-separated connector strings into individual entries."""
    if connector and connector.strip() and ',' in connector:
        return [c.strip() for c in connector.split(',')]
    elif connector and connector.strip():
        return [connector.strip()]
    return []

# Process all connector columns into a flat list
data_processed = []
for _, row in data.iterrows():
    for column in connector_columns:
        for conn in split_connector(row[column]):
            if conn:
                data_processed.append({'year': row['year'], 'connector_type': conn})

data_processed_df = pd.DataFrame(data_processed)
data_processed_df['connector_cluster'] = data_processed_df['connector_type'].apply(cluster_connector)

# Cumulative count by cluster and year
cluster_by_year = (
    data_processed_df.groupby(['year', 'connector_cluster']).size()
    .unstack().fillna(0).cumsum()
)
cluster_by_year = cluster_by_year[cluster_by_year.index >= 2009]

plt.rcParams['font.size'] = 14
ax = cluster_by_year.plot(kind='bar', stacked=True)
plt.xlabel('Year')
plt.ylabel('Number of Connectors')
plt.legend(title='Connector Type')
plt.savefig('../results/figures/statistics/connector_type_clusters.png', bbox_inches='tight', dpi=150)
plt.show()